# Assignment 2: End-to-End Machine Learning Pipeline
**Course/Topic:** Machine Learning Lifecycle & Pipeline Development  
**Business Goal:** Predicting High Investment Risk for Financial Customers

---

## Business Problem Understanding
In the financial services industry, wealth managers and investment advisors must understand their clients' risk profiles. This is crucial for:
1. **Regulatory Compliance:** Know Your Customer (KYC) regulations mandate that advisors assess the suitability of investment products.
2. **Client Retention & Trust:** Recommending high-risk portfolios to conservative investors can lead to massive financial losses and loss of trust.
3. **Risk Management:** Helping clients build balanced portfolios that align with their age, experience, and wealth.

In this notebook, we build an **End-to-End Machine Learning Pipeline** to predict whether a customer is a **High Investment Risk** client (`High_Investment_Risk = Yes/No`) based on their demographic, financial, and behavioral attributes.

---
## Pipeline Steps:
1. **Load and Inspect the Dataset**
2. **Define Features (X) and Target Variable (y)**
3. **Handle Data Cleaning and Missing Values**
4. **Preprocess Categorical and Numerical Features**
5. **Split Data into Training and Testing Sets**
6. **Train a Baseline Logistic Regression Model**
7. **Evaluate Model Performance**
8. **Improvement Step: Handle Class Imbalance (Balanced Logistic Regression)**
9. **Task 2 Explanation and Summary**

In [ ]:
# Step 0: Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Set visual style for plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Inspect the Dataset
We load the dataset `financial_customer_investment_risk_dataset (1).xls`. Note that although the file has a `.xls` extension, it is structured as a comma-separated values (CSV) file. We will use `pd.read_csv()` to load it.

Let's inspect the dimensions, headers, data types, and check for missing values.

In [ ]:
# Load dataset
file_path = 'financial_customer_investment_risk_dataset (1).xls'
df = pd.read_csv(file_path)

# 1. Print Shape
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")

# 2. Display first 5 rows
print("--- First 5 Rows ---")
display(df.head())

# 3. Inspect column types and non-null values
print("\n--- Dataset Info ---")
df.info()

# 4. Describe statistical summaries
print("\n--- Summary Statistics of Numerical Variables ---")
display(df.describe())

# 5. Check for missing values
print("\n--- Missing Values Count Per Column ---")
print(df.isnull().sum())

### Exploratory Data Analysis & Visualizations
Let's analyze:
1. The class balance of our target variable `High_Investment_Risk`.
2. The relation between age, income, and high investment risk.
3. The correlations among numerical columns.

In [ ]:
# Plot 1: Target Variable Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='High_Investment_Risk', palette='Set2')
plt.title('Distribution of Target Variable (High Investment Risk)')
plt.xlabel('High Investment Risk')
plt.ylabel('Count')
plt.show()

# Print class counts and percentages
counts = df['High_Investment_Risk'].value_counts()
pcts = df['High_Investment_Risk'].value_counts(normalize=True) * 100
for idx in counts.index:
    print(f"Class '{idx}': {counts[idx]} samples ({pcts[idx]:.2f}%)")

In [ ]:
# Plot 2: Age vs Annual Income colored by High Investment Risk
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='Age', y='Annual_Income', hue='High_Investment_Risk', palette='Set1', alpha=0.8)
plt.title('Age vs Annual Income by Risk Category')
plt.xlabel('Age (Years)')
plt.ylabel('Annual Income ($)')
plt.legend(title='High Risk?')
plt.show()

In [ ]:
# Plot 3: Correlation matrix of numerical features
plt.figure(figsize=(10, 8))
# Exclude Customer_ID as it is just an identifier
numeric_cols = df.select_dtypes(include=[np.number]).drop(columns=['Customer_ID'], errors='ignore')
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

## 2. Define Features (X) and Target Variable (y)
We will:
- Set `y` to `High_Investment_Risk` (mapping `Yes` to 1, and `No` to 0).
- Set `X` to contain all columns except `Customer_ID` (uninformative unique identifier) and `High_Investment_Risk` (our target).

In [ ]:
# Map target variable to binary values (0 and 1)
y = df['High_Investment_Risk'].map({'Yes': 1, 'No': 0})

# Drop irrelevant features and target
X = df.drop(columns=['Customer_ID', 'High_Investment_Risk'])

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

## 3. Split the Data into Training and Testing Sets
To ensure that we evaluate our model on unseen data, we split the dataset.
- **Train set:** 80%
- **Test set:** 20%
- **Stratification:** Because `High_Investment_Risk` has high class imbalance (only ~5% Yes), we must use `stratify=y` to ensure that training and testing sets maintain the same proportion of high-risk customers.

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Testing set:  X_test = {X_test.shape}, y_test = {y_test.shape}")
print("\nTrain set label distribution:\n", y_train.value_counts(normalize=True))
print("Test set label distribution:\n", y_test.value_counts(normalize=True))

## 4. Build the Preprocessing Pipeline
To prevent data leakage, we structure our cleaning and preprocessing steps in a Scikit-Learn `Pipeline`.
- **Numerical columns (`Age`, `Annual_Income`, etc.)**:
  - Impute missing values with the **median** (robust to outliers).
  - Scale features using **StandardScaler** to ensure all numerical features have a mean of 0 and variance of 1.
- **Categorical columns (`Employment_Status`, `Risk_Tolerance`, etc.)**:
  - Impute missing values with the **most frequent** value (mode).
  - Encode categorical variables using **OneHotEncoder** (converting them into dummy/binary columns).

We bundle these steps using `ColumnTransformer`.

In [ ]:
# Identify columns by type
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

# Define pipelines for numerical and categorical preprocessing
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine columns using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

## 5. Train a Simple Baseline Model Using Logistic Regression
We combine our `preprocessor` and a standard `LogisticRegression` classifier into an end-to-end pipeline, fit it on the training data, and generate predictions for the test set.

In [ ]:
# Create end-to-end baseline pipeline
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Train model
baseline_pipeline.fit(X_train, y_train)

# Predict on test data
y_pred_baseline = baseline_pipeline.predict(X_test)

## 6. Evaluate the Baseline Model
Let's compute the confusion matrix, accuracy, and detailed classification report.

In [ ]:
# Calculate performance metrics
acc_baseline = accuracy_score(y_test, y_pred_baseline)
cm_baseline = confusion_matrix(y_test, y_pred_baseline)

print("=== Baseline Logistic Regression Metrics ===")
print(f"Accuracy Score: {acc_baseline:.4f}")
print("\nConfusion Matrix:")
print(cm_baseline)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_baseline, zero_division=0))

# Visualize Confusion Matrix
plt.figure(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_baseline, display_labels=['No Risk (0)', 'High Risk (1)'])
disp.plot(cmap='Blues', ax=plt.gca())
plt.grid(False)
plt.title('Baseline LR Confusion Matrix')
plt.show()

### Analysis of Baseline Model
As shown in the classification report, the baseline model achieves an accuracy of **93.59%**. However, looking closer at the metrics for class `1` (High Risk):
- **Precision:** 0.00
- **Recall:** 0.00
- **F1-Score:** 0.00
- **Confusion Matrix:** `[[73, 1], [4, 0]]`

Because the dataset is heavily skewed (95% No Risk, 5% High Risk), the standard Logistic Regression model achieves high accuracy simply by predicting `No Risk` for almost every customer. It only predicted 1 customer as High Risk, and that prediction was incorrect (False Positive). It missed all 4 high-risk customers in the test set (False Negatives). This is a critical business failure because we fail to identify any of the high-risk clients!

---

## 7. Advanced Step: Train a Balanced Logistic Regression Model
To address the class imbalance, we can train a Logistic Regression model with the parameter `class_weight='balanced'`. This penalizes misclassifications on the minority class (`Yes`) proportionally to its under-representation.

In [ ]:
# Define balanced pipeline
balanced_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])

# Train balanced model
balanced_pipeline.fit(X_train, y_train)

# Predict on test data
y_pred_balanced = balanced_pipeline.predict(X_test)

# Calculate performance metrics
acc_balanced = accuracy_score(y_test, y_pred_balanced)
cm_balanced = confusion_matrix(y_test, y_pred_balanced)

print("=== Balanced Logistic Regression Metrics ===")
print(f"Accuracy Score: {acc_balanced:.4f}")
print("\nConfusion Matrix:")
print(cm_balanced)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced, zero_division=0))

# Visualize Confusion Matrix
plt.figure(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_balanced, display_labels=['No Risk (0)', 'High Risk (1)'])
disp.plot(cmap='Greens', ax=plt.gca())
plt.grid(False)
plt.title('Balanced LR Confusion Matrix')
plt.show()

### Analysis of Balanced Model
With the balanced class weights:
- **Accuracy:** Drops slightly to **92.31%**.
- **Confusion Matrix:** `[[70, 4], [2, 2]]`
- **Recall (Class 1):** Increases from **0.00** to **0.50** (we now successfully identify 50% of high-risk clients, i.e., 2 out of 4).
- **F1-Score (Class 1):** Increases from **0.00** to **0.50** (or 0.40 depending on exact thresholds/precision).

This demonstrates the importance of choosing the right evaluation metrics and model configurations when dealing with highly imbalanced real-world datasets.

---

## Task 2: Explain and Submit Your Work

1. **What your model is trying to predict:**
   - The model is trying to predict whether a client is in the **High Investment Risk** category (`High_Investment_Risk` = `Yes` (1) or `No` (0)). This target column indicates if a customer's demographic and financial profile makes them a highly risky investor.

2. **What dataset you used:**
   - We used the **Financial Customer Investment Risk Dataset** (`financial_customer_investment_risk_dataset (1).xls`). It is a CSV file containing 387 client profiles with 13 features.

3. **What features and target you selected:**
   - **Target variable:** `High_Investment_Risk`
   - **Uninformative dropped variable:** `Customer_ID`
   - **Features (X) selected:** 
     - *Numerical (7):* `Age`, `Annual_Income`, `Investment_Experience_Years`, `Portfolio_Value`, `Number_of_Investment_Products`, `Market_Volatility_Concern`, `Financial_Literacy_Score`
     - *Categorical (4):* `Employment_Status`, `Risk_Tolerance`, `Advisor_Contacted`, `Previous_Investment_Loss`

4. **What result you obtained:**
   - **Baseline Logistic Regression:** Achieved an **Accuracy of 93.59%**, but had a **Recall of 0%** and **F1-Score of 0%** for the High Risk category because it failed to correctly predict any high-risk customers due to class imbalance (predicting only 1 incorrect customer as positive).
   - **Balanced Logistic Regression:** Achieved a slightly lower **Accuracy of 92.31%**, but successfully predicted **50.00% of high-risk customers (Recall = 0.50)** and achieved a **Precision of 33.33%** and an **F1-Score of 40.00%** on the high-risk class.

5. **One limitation of your model or dataset:**
   - **Extreme Class Imbalance & Small Dataset:** The dataset contains only 387 total rows, out of which only 20 are positive (`High_Investment_Risk = Yes`). With only 20 positive examples overall (and only 4 in the test set), the model has very little data to learn what characteristics distinguish a high-risk client from a low-risk client. As a result, the evaluation metrics on the test set are highly volatile and the model has low positive-class precision (high rate of false positives when forced to predict class 1). To resolve this, we would need to collect more data, especially from high-risk individuals, or apply techniques like SMOTE (Synthetic Minority Over-sampling Technique).